In [22]:
# import numpy as np

# # the "dataset": an 8x8 image with a vertical edge at column 4, built explicitly
# image = np.zeros((8, 8))
# image[:, 4:] = 1.0

# # a hand-designed vertical-edge kernel: negative left, positive right
# vertical_edge_kernel = np.array([
#     [-1.0, 0.0, 1.0],
#     [-1.0, 0.0, 1.0],
#     [-1.0, 0.0, 1.0],
# ])

# def conv2d(img, kernel, stride=1, padding=0):
#     if padding > 0:
#         img = np.pad(img, padding, mode="constant", constant_values=0.0)
#     h, w = img.shape
#     kh, kw = kernel.shape
#     out_h = (h - kh) // stride + 1
#     out_w = (w - kw) // stride + 1
#     out = np.zeros((out_h, out_w))
#     for i in range(out_h):
#         for j in range(out_w):
#             row, col = i * stride, j * stride
#             patch = img[row:row + kh, col:col + kw]
#             out[i, j] = np.sum(patch * kernel)   # the dot product
#     return out

In [20]:
image

array([[0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.]])

In [21]:
conv2d(image, vertical_edge_kernel)

array([[0., 0., 3., 3., 0., 0.],
       [0., 0., 3., 3., 0., 0.],
       [0., 0., 3., 3., 0., 0.],
       [0., 0., 3., 3., 0., 0.],
       [0., 0., 3., 3., 0., 0.],
       [0., 0., 3., 3., 0., 0.]])

In [4]:
image

array([[0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.],
       [0., 0., 0., 0., 1., 1., 1., 1.]])

In [6]:


IMG_SIZE = 16

def draw_square(rng):
    img = np.zeros((IMG_SIZE, IMG_SIZE))
    size = rng.integers(6, 11)
    top, left = rng.integers(0, IMG_SIZE - size), rng.integers(0, IMG_SIZE - size)
    img[top:top + size, left:left + size] = 1.0
    return img

def draw_circle(rng):
    img = np.zeros((IMG_SIZE, IMG_SIZE))
    radius = rng.integers(4, 7)
    cy, cx = rng.integers(radius, IMG_SIZE - radius), rng.integers(radius, IMG_SIZE - radius)
    yy, xx = np.ogrid[:IMG_SIZE, :IMG_SIZE]
    img[(yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2] = 1.0
    return img

def make_dataset(n_per_class=200, seed=42):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for _ in range(n_per_class):
        images.append(draw_square(rng)); labels.append(0)
        images.append(draw_circle(rng)); labels.append(1)
    images = np.stack(images).astype(np.float32)[:, None, :, :]   # [N,1,16,16]
    labels = np.array(labels, dtype=np.int64)
    order = rng.permutation(len(labels))
    return images[order], labels[order]

In [18]:
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(16 * 4 * 4, num_classes)   # 256 -> 2, matches the shape check above

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # [B,1,16,16] -> [B,8,8,8]
        x = self.pool(self.relu(self.conv2(x)))   # [B,8,8,8]   -> [B,16,4,4]
        x = x.flatten(start_dim=1)             # [B,16,4,4]  -> [B,256]
        return self.fc(x)        

In [28]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

images, labels = make_dataset(n_per_class=200, seed=42)
x, y = torch.from_numpy(images), torch.from_numpy(labels)
split = int(0.8 * len(y))
train_loader = DataLoader(TensorDataset(x[:split], y[:split]), batch_size=32, shuffle=True)

model = SimpleCNN(num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)



"""



"""
for epoch in range(1, 11):
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0) 
        correct += (logits.argmax(dim=1) == labels).sum().item()
        n += images.size(0)
    print(f"epoch {epoch:2d}  loss {total_loss/n:.4f}  train acc {correct/n:.3f}")

epoch  1  loss 0.6925  train acc 0.491
epoch  2  loss 0.6818  train acc 0.531
epoch  3  loss 0.6726  train acc 0.659
epoch  4  loss 0.6542  train acc 0.628
epoch  5  loss 0.6329  train acc 0.647
epoch  6  loss 0.6079  train acc 0.703
epoch  7  loss 0.5816  train acc 0.741
epoch  8  loss 0.5523  train acc 0.762
epoch  9  loss 0.5189  train acc 0.850
epoch 10  loss 0.4844  train acc 0.838


In [27]:
x

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 1., 1., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        ...,


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0.

In [26]:
split

320